In [ ]:
install.packages("data.table")
install.packages("seqinr")
install.packages("openxlsx")
if (!require("BiocManager", quietly = TRUE))
install.packages("BiocManager")
BiocManager::install("IRanges")
BiocManager::install("GenomicRanges")
BiocManager::install("GenomicAlignments")
BiocManager::install("Rsamtools")
BiocManager::install("Biostrings")
BiocManager::install("GenomeInfoDb")
BiocManager::install("BSgenome")

: 

In [ ]:
install.packages("/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/workflow/scripts/PICB",repos=NULL, type="source")

In [ ]:
library('PICB')

In [ ]:
uniqueonly="seeds"
uniqueandprimary="cores"
allalignments="clusters"

In [ ]:
#' @param FASTA.NAME path to the fasta file
#'
#' @return SeqInfo object with all chromosome names and lengths from the fasta file
#' @export
#' @author Aleksandr Friman
#'
#' @examples mySI<-PICBloadfasta("~/path/to/your.fasta")
FASTA.NAME <- "/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/results/strain_genome/C57BL_6NJ.fasta" 
PICBloadfasta<-function(FASTA.NAME = NULL){
if(is.null(FASTA.NAME)) stop("Please provide FASTA.NAME !")
FAdata<-seqinr::read.fasta(FASTA.NAME)
FAnames<-names(FAdata)
FAlengths<-lengths(FAdata)
return(GenomeInfoDb::Seqinfo(seqnames = FAnames, seqlengths = FAlengths))
}

In [ ]:
#' @param BAMFILE name of the bam file to load. Should be sorted and indexed.
#' @param REFERENCE.GENOME name of genome. For example "BSgenome.Dmelanogaster.UCSC.dm6"
#' @param SIMPLE.CIGAR simpleCigar parameter of Rsamtools::ScanBamParam
#' @param IS.SECONDARY.ALIGNMENT defines loading of primary/secondary alignments. Default value NA loads both primary and secondary.
#' @param STANDARD.CONTIGS.ONLY use only standard chromosomes
#' @param PERFECT.MATCH.ONLY load only alignments without mismatches
#' @param FILTER.BY.FLAG enables filtering by flag. TRUE by default.
#' @param SELECT.FLAG vector of flags to use. Default value c(0,16, 272, 256).
#' @param USE.SIZE.FILTER enables filter by alignment size. True by default.
#' @param READ.SIZE.RANGE allowed alignment sizes. c(18,50) by default.
#' @param TAGS tags to import from bam file. c("NH","NM") by default.
#' @param WHAT "what" parameter of Rsamtools::ScanBamParam. c("flag") by default.
#' @param SEQ.LEVELS.STYLE style of chromosome names for BSgenome. "UCSC" by default.
#' @param GET.ORIGINAL.SEQUENCE adds "seq" to WHAT. False by default.
#' @param VERBOSE enables progress output. True by default.
#'
#' @author Pavol Genzor
#' @author Daniel Stoyko
#' @author Aleksandr Friman
#' @return list of Granges objects named "unique" for unique mapping alignments,
#' "multi.primary" for primary multimapping alignments,
#' "multi.secondary" for secondary multimapping alignments
#' @export
#'
#' @examples PICBload(BAMFILE="~/example-piRNA-library.bam",
#REFERENCE.GENOME="BSgenome.Dmelanogaster.UCSC.dm6")
PICBload <- function(
## INPUTS
BAMFILE=NULL,
REFERENCE.GENOME=NULL,
## OPTIONS
SIMPLE.CIGAR=TRUE,
IS.SECONDARY.ALIGNMENT=NA,
STANDARD.CONTIGS.ONLY=TRUE,
PERFECT.MATCH.ONLY=FALSE,
FILTER.BY.FLAG=TRUE,
SELECT.FLAG=c(0,16, 272, 256),
USE.SIZE.FILTER=TRUE,
READ.SIZE.RANGE=c(18,50),
TAGS=c("NH","NM"),
WHAT=c("flag"),
## EXTRA OPTIONS
SEQ.LEVELS.STYLE="UCSC",
GET.ORIGINAL.SEQUENCE=FALSE,
VERBOSE=TRUE){
## NOTE ON BAM INFO
## Flag: 256 = not primary alignment; 272 = reverse strand not primary alignment;
## Flag: 0 = forward unpaired unique alignment; 16 = reverse unpaired unique alignment
## Tags: NH:i:1 = unique alignment; NM = edit distance to the reference
## libraries
#suppressPackageStartupMessages({library("Rsamtools");
# library("GenomicAlignments");
#})
outputAlignments<-list()
BAMFILE <- "/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/results/STAR_rna_strain_wise/C57BL_6NJ-16.5dpc.1/Aligned.sortedByCoord.out.bam"
REFERENCE.GENOME <- "/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/results/strain_genome/C57BL_6NJ.fasta" 
## check input
if(is.null(BAMFILE)) stop("Please provide full path to a .bam file !!!")
if(is.null(REFERENCE.GENOME)) stop("Please provide REFERENCE.GENOME")
if(isTRUE(GET.ORIGINAL.SEQUENCE)){WHAT=c(WHAT,"seq")}
## for report
if (VERBOSE) message("Processing ...")
justPrimaryOrSecondary<-function(IS.SECONDARY.ALIGNMENT){
## PARAMETERS FOR LOADING BAM FILE
if(isTRUE(STANDARD.CONTIGS.ONLY)){
if (typeof(REFERENCE.GENOME)=="character"){
SI<-PICBgetchromosomes(REFERENCE.GENOME, SEQ.LEVELS.STYLE)
}else{
SI<-REFERENCE.GENOME
}
REG.CHR <- GenomeInfoDb::seqnames(SI)
BAM.FILE.HEADER<-Rsamtools::BamFile(BAMFILE)
BAM.FILE.CHR<-GenomeInfoDb::seqnames(GenomeInfoDb::seqinfo(BAM.FILE.HEADER))
REG.CHR<-REG.CHR[REG.CHR %in% BAM.FILE.CHR]
WHICH = GenomicRanges::GRanges(seqnames=REG.CHR,
ranges=IRanges::IRanges(start=rep(1, length(REG.CHR)),
end=GenomeInfoDb::seqlengths(SI)),
strand=rep("*", length(REG.CHR)))
PARAM = Rsamtools::ScanBamParam(flag = Rsamtools::scanBamFlag(isUnmappedQuery =
FALSE,
isSecondaryAlignment
= IS.SECONDARY.ALIGNMENT),
tag = TAGS, simpleCigar = SIMPLE.CIGAR, what =
WHAT, which = WHICH)
}else{
PARAM = Rsamtools::ScanBamParam(flag = Rsamtools::scanBamFlag(isUnmappedQuery =
FALSE,
isSecondaryAlignment
= IS.SECONDARY.ALIGNMENT),
tag = TAGS, simpleCigar = SIMPLE.CIGAR, what =
WHAT)
}
if (VERBOSE){
message("\nprepared loading parameters")
message(paste0("\tTAGS:\t",paste(TAGS, collapse = ", ")))
message(paste0("\tCIGAR:\t",ifelse(isTRUE(SIMPLE.CIGAR),"simple cigar","all
cigar")))
message(paste0("\tWHAT:\t",paste0(WHAT,collapse = ", ")))
message("loading .bam file into GAlignemnts")
if(is.na(IS.SECONDARY.ALIGNMENT)){
message("\n******"); message("SLOW - Loadding all reads");
message(" => to load unique and primary alignments set")
message(" => IS.SECONDARY.ALIGNMENT=FALSE"); message("******\n")
} else if (!IS.SECONDARY.ALIGNMENT){
message("\n******"); message("Loading unique and primary alignments !!!")
message("******\n")
} else if (IS.SECONDARY.ALIGNMENT){
message("\n******"); message("Loading secondary alignments only !!!")
message("******\n")
}
}
GA <- GenomicAlignments::readGAlignments(file = BAMFILE,
use.names = TRUE,
param = PARAM)
#checking the tags consistency
for (tagcheck in TAGS){
if (any(is.na(GenomicRanges::mcols(GA)[[tagcheck]]))){
stop(paste0("Tag ", tagcheck, " contains NA values. Check your bam file."))
}
}
## ***
GA.IN <- length(GA)
if (VERBOSE) message(paste0("\tIMPORTED: ", GA.IN))
if(isTRUE(USE.SIZE.FILTER)){
if (VERBOSE){
message("\nfiltering by read size")
message(paste0("\tRANGE:\t",
paste(min(READ.SIZE.RANGE),
max(READ.SIZE.RANGE),
sep = "-")))
}
GA <- GA[GenomicAlignments::width(GA) %in% seq(min(READ.SIZE.RANGE),max(READ.SIZE.RANGE),by = 1)]
## ***
REMAINDER = (length(GA)/GA.IN)*100
if (VERBOSE) message(paste0("\tREMAINDER: ", round(REMAINDER, digits = 2) )) }
if(isTRUE(FILTER.BY.FLAG)){
if (VERBOSE) message("\nfiltering based on flags")
GARP <- GA[GenomicRanges::mcols(GA)[["flag"]] %in% SELECT.FLAG]
#mcols(GARP)[["flag"]] <- NULL
## ***
REMAINDER = (length(GARP)/GA.IN)*100
if (VERBOSE) message(paste0("\tREMAINDER: ", round(REMAINDER, digits = 2) )) }
else { GARP <- GA }
if(isTRUE(PERFECT.MATCH.ONLY)){
if (VERBOSE) message("\nremoving reads with mismatches")
GARP <- GARP[GenomicRanges::mcols(GARP)[["NM"]] %in% c(0)]
## ***
REMAINDER = (length(GARP)/GA.IN)*100
if (VERBOSE) message(paste0("\tREMAINDER: ", round(REMAINDER, digits = 2) )) }
if(isTRUE(GET.ORIGINAL.SEQUENCE)){
if (VERBOSE) message("\nretrieving original read sequences")
BAMSEQ <- GenomicRanges::mcols(GARP)[["seq"]]
ISONMINUS <- as.logical(GenomicAlignments::strand(GARP) == "-")
BAMSEQ[ISONMINUS] <- Biostrings::reverseComplement(BAMSEQ[ISONMINUS])
GenomicRanges::mcols(GARP)[["seq"]] <- BAMSEQ }
if (VERBOSE) message("\nconverting to GRanges")
GARP.GR <- GenomicRanges::granges(GARP, use.names = TRUE, use.mcols = TRUE)
if(!SEQ.LEVELS.STYLE %in% "UCSC"){
if (VERBOSE) message(paste0("\nchanging seqlevels style to: ",SEQ.LEVELS.STYLE))
GenomeInfoDb::seqlevelsStyle(GARP.GR) <- SEQ.LEVELS.STYLE
}
return(GARP.GR)
}
if (VERBOSE){
message("\nSorting into uniquemappers vs multimappers and primary vs secondary
alignments")
}
if (is.na(IS.SECONDARY.ALIGNMENT)){#justPrimaryOrSecondary
if (VERBOSE) message("Loading primary only")
PrimaryAlignments<-justPrimaryOrSecondary(IS.SECONDARY.ALIGNMENT=FALSE)
outputAlignments[['unique']]<-
PrimaryAlignments[GenomicRanges::mcols(PrimaryAlignments)[['NH']]==1]
outputAlignments[['multi.primary']]<-
PrimaryAlignments[(GenomicRanges::mcols(PrimaryAlignments)[['NH']]>1)]
if (VERBOSE) message("Loading secondary only")
outputAlignments[['multi.secondary']]<-
justPrimaryOrSecondary(IS.SECONDARY.ALIGNMENT=TRUE)
}else if(IS.SECONDARY.ALIGNMENT){
outputAlignments[['unique']]<-NULL
outputAlignments[['multi.primary']]<-NULL
if (VERBOSE) message("Loading secondary only")
outputAlignments[['multi.secondary']]<- justPrimaryOrSecondary(IS.SECONDARY.ALIGNMENT=TRUE)
}else if(!IS.SECONDARY.ALIGNMENT){
if (VERBOSE) message("Loading primary only")
PrimaryAlignments<-justPrimaryOrSecondary(IS.SECONDARY.ALIGNMENT=FALSE)
outputAlignments[['unique']]<-
PrimaryAlignments[GenomicRanges::mcols(PrimaryAlignments)[['NH']]==1]
outputAlignments[['multi.primary']]<-
PrimaryAlignments[(GenomicRanges::mcols(PrimaryAlignments)[['NH']]>1)]
outputAlignments[['multi.secondary']]<-NULL
}
## RETURN
if (VERBOSE){
message("\nDone!")
message("")
}
return(outputAlignments)
}

In [ ]:
#' @param IN.ALIGNMENTS list of alignments from PICBload
#' @param REFERENCE.GENOME name of genome. For example "BSgenome.Dmelanogaster.UCSC.dm6"
#' @param UNIQUEMAPPERS.SLIDING.WINDOW.WIDTH width of sliding window for unique mappers. 350 nt by default
#' @param UNIQUEMAPPERS.SLIDING.WINDOW.STEP step of sliding windows for unique mappers. width/10 by default
#' @param PRIMARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH width of sliding window for primary multimapping alignments. 350 nt by default
#' @param PRIMARY.MULTIMAPPERS.SLIDING.WINDOW.STEP step of sliding windows for primary multimapping alignments. width/10 by default
#' @param SECONDARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH width of sliding window for secondary multimapping alignments. 1000 nt by default
#' @param SECONDARY.MULTIMAPPERS.SLIDING.WINDOW.STEP step of sliding windows for secondary multimapping alignments. width/10 by default
#' @param LIBRARY.SIZE number of reads in the library. By default computed as number of unique mapping alignments + number of primary multimapping alignments.
#' @param MIN.UNIQUE.ALIGNMENTS.PER.WINDOW absolute number of unique mapping alignments per window to call it. By default computed as 2 FPKM.
#' @param MIN.UNIQUE.SEQUENCES.PER.WINDOW absolute number of unique mapping sequences per window to call it. By default computed as width/50.
#' @param MIN.PRIMARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW absolute number of primary multimapping alignments per window to call it. By default computed as 4 FPKM.
#' @param MIN.SECONDARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW absolute number of secondary multimapping alignments per window to call it. By default computed as 0.2 FPKM.
#' @param MIN.SEED.LENGTH minimum length of a seed. By default computed as 2x unique mapper window size + 100.
#' @param MIN.COVERED.SEED.LENGTH minimum number of seed nucleotides covered by unique mappers. 0 by default.
#' @param THRESHOLD.SEEDS.GAP minimum gap between seeds to not merge them. 0 by default.
#' @param THRESHOLD.CORES.GAP minimum gap between cores to not merge them. 0 by default.
#' @param THRESHOLD.CLUSTERS.GAP minimum gap between clusters to not merge them. 0 by default.
#' @param SEQ.LEVELS.STYLE style of chromosome names for BSgenome. "UCSC" by default.
#' @param MIN.OVERLAP minimum overlap between seeds and cores, as well as between cores and clusters 5 nt by default.
#' @param PROVIDE.NON.NORMALIZED include non-normalized to the library size statistics in the output annotations
#' @param VERBOSITY verbosity level 0/1/2/3. 2 by default.
#'
#' @author Pavol Genzor
#' @author Daniel Stoyko
#' @author Aleksandr Friman
#' @return list of annotated Granges objects named "seeds" for seeds,
#' "cores" for cores,
#' "clusters" for clusters
#' @export
#'
#' @examples PICBbuild(IN.ALIGNMENTS = myAlignmentsFromPIBCload,
REFERENCE.GENOME= "/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/results/strain_genome/C57BL_6NJ.fasta" 
PICBbuild <-
function(
## MAIN
IN.ALIGNMENTS,
REFERENCE.GENOME,
## OPTIONS
UNIQUEMAPPERS.SLIDING.WINDOW.WIDTH=350,
UNIQUEMAPPERS.SLIDING.WINDOW.STEP=round(UNIQUEMAPPERS.SLIDING.WINDOW.WIDTH/10,0),
PRIMARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH=350,
PRIMARY.MULTIMAPPERS.SLIDING.WINDOW.STEP=round(PRIMARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH/10,0),
SECONDARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH=1000,
SECONDARY.MULTIMAPPERS.SLIDING.WINDOW.STEP=round(SECONDARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH/10,0),
LIBRARY.SIZE=length(IN.ALIGNMENTS$unique)+length(IN.ALIGNMENTS$multi.primary),
MIN.UNIQUE.ALIGNMENTS.PER.WINDOW=2*(UNIQUEMAPPERS.SLIDING.WINDOW.WIDTH/1e3)*(LIBRARY.SIZE/1e6),
MIN.UNIQUE.SEQUENCES.PER.WINDOW=min(MIN.UNIQUE.ALIGNMENTS.PER.WINDOW,
round(UNIQUEMAPPERS.SLIDING.WINDOW.WIDTH/50,0)),
MIN.PRIMARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW=4*
(PRIMARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH/1e3)*(LIBRARY.SIZE/1e6),
MIN.SECONDARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW=0.2*
(SECONDARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH/1e3)*(LIBRARY.SIZE/1e6),
MIN.SEED.LENGTH=2*UNIQUEMAPPERS.SLIDING.WINDOW.WIDTH+100,
MIN.COVERED.SEED.LENGTH=0,
## CLUSTER FILTERS
THRESHOLD.SEEDS.GAP=0, THRESHOLD.CORES.GAP=0, THRESHOLD.CLUSTERS.GAP=0,
## EXTRA OPTIONS
SEQ.LEVELS.STYLE="UCSC",
MIN.OVERLAP=5,
PROVIDE.NON.NORMALIZED = FALSE,
VERBOSITY=2){
## Authors: Pavol Genzor, Alex Friman, Daniel Stoyko
## Usage: Build piRNA clusters using piRNA sequencing data
## OUTPUT FOR DEBUGING PURPOSES
if (VERBOSITY>2) {
tmpENV<-as.list(environment())
tmpENV$IN.ALIGNMENTS<-NULL
print(tmpENV)
}
IN.ALIGNMENTS <- "/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/results/STAR_rna_strain_wise/C57BL_6NJ-16.5dpc.1/Aligned.sortedByCoord.out.bam"
REFERENCE.GENOME <- "/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/results/strain_genome/C57BL_6NJ.fasta" 
## Check input
if(is.null(IN.ALIGNMENTS)) stop("Please provide IN.ALIGNMENTS !")
if(is.null(REFERENCE.GENOME)) stop("Please provide REFERENCE.GENOME !")

for (columnName in c("unique", "multi.primary", "multi.secondary")){
if(!"NH" %in% colnames(GenomicRanges::mcols(IN.ALIGNMENTS[[columnName]])))
stop("The IN.ALIGNMENTS must contain NH information !")
}
if (VERBOSITY>0) message(paste("PICB v", packageVersion("PICB"),"Starting ... "))
## KEEP STANDARD CHROMOSOMES
if (typeof(REFERENCE.GENOME)=="character"){
SI<-PICBgetchromosomes(REFERENCE.GENOME, SEQ.LEVELS.STYLE)
}else{
SI<-REFERENCE.GENOME
}
## Clean data
if (VERBOSITY>1) message("\n\tKeeping standard linear chromosomes")
for (columnName in c("unique", "multi.primary", "multi.secondary")){
#trying to change the seqlevels style.
#read https://github.com/Bioconductor/GenomeInfoDb/blob/devel/inst/extdata/dataFiles/README
#for more info
tryCatch( expr = {GenomeInfoDb::seqlevelsStyle(IN.ALIGNMENTS[[columnName]]) <-
SEQ.LEVELS.STYLE},
error = function(e){
print(e)
print("Failed to change SEQ.LEVELS.STYLE")
print("readhttps://github.com/Bioconductor/GenomeInfoDb/blob/devel/inst/extdata/dataFiles/README")
print("Continuing hoping for the best")
})
KEEP.SEQLEVELS <- GenomeInfoDb::seqlevels(SI)
KEEP.SEQLEVELS <- KEEP.SEQLEVELS[KEEP.SEQLEVELS %in%
GenomeInfoDb::seqlevels(IN.ALIGNMENTS[[columnName]])]
IN.ALIGNMENTS[[columnName]] <- GenomeInfoDb::keepSeqlevels(x =
IN.ALIGNMENTS[[columnName]], value = KEEP.SEQLEVELS, pruning.mode = "coarse")
}
WGRU<-IN.ALIGNMENTS$unique
WGRMP<-IN.ALIGNMENTS$multi.primary
if (VERBOSITY>0) message("\nBuilding ... STEP 1... Searching using windows\n")
## Make genome sliding window for unique mappers
if (VERBOSITY>1) message("\tSliding window analysis")
if (VERBOSITY>1) message(paste0("\t\tUNIQUE MAPPERS\n\t\tWINDOW:
",UNIQUEMAPPERS.SLIDING.WINDOW.WIDTH,"\n\t\tSTEP: ", UNIQUEMAPPERS.SLIDING.WINDOW.STEP))
AG.gr <-
GenomicRanges::GRanges(data.table::data.table("chr"=GenomicRanges::seqnames(SI),"start"=
1,"end"=GenomeInfoDb::seqlengths(SI),"strand"="*"))
AG.sw.uniq <- unlist(GenomicRanges::slidingWindows(x = AG.gr, width =
UNIQUEMAPPERS.SLIDING.WINDOW.WIDTH, step = UNIQUEMAPPERS.SLIDING.WINDOW.STEP))
## Make genome sliding window for multi mappers
if (VERBOSITY>1) message(paste0("\t\tPRIMARY MULTI MAPPERS\n\t\tWINDOW:
",PRIMARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH,"\n\t\tSTEP: ",
PRIMARY.MULTIMAPPERS.SLIDING.WINDOW.STEP))
#AG.gr <-
GRanges(data.table("chr"=seqnames(SI),"start"=1,"end"=seqlengths(SI),"strand"="*"))
AG.sw.primary.mult <- unlist(GenomicRanges::slidingWindows(x = AG.gr, width =
PRIMARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH, step =
PRIMARY.MULTIMAPPERS.SLIDING.WINDOW.STEP))
if (VERBOSITY>1) message(paste0("\t\tSECONDARY MULTI MAPPERS\n\t\tWINDOW:
",SECONDARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH,"\n\t\tSTEP: ",
SECONDARY.MULTIMAPPERS.SLIDING.WINDOW.STEP))
AG.sw.secondary.mult <- unlist(GenomicRanges::slidingWindows(x = AG.gr, width =
SECONDARY.MULTIMAPPERS.SLIDING.WINDOW.WIDTH, step =
SECONDARY.MULTIMAPPERS.SLIDING.WINDOW.STEP))
## Counting piRNAs per window: Uniq and all multi
GenomicRanges::mcols(AG.sw.uniq)[["uniq_piRNA_plus"]] <-
GenomicRanges::countOverlaps(query = AG.sw.uniq, subject =
WGRU[GenomicRanges::strand(WGRU) == "+"])
GenomicRanges::mcols(AG.sw.uniq)[["uniq_piRNA_minus"]] <-
GenomicRanges::countOverlaps(query = AG.sw.uniq, subject =
WGRU[GenomicRanges::strand(WGRU) == "-"])
GenomicRanges::mcols(AG.sw.uniq)[["uniq_intervals_plus"]] <-
GenomicRanges::countOverlaps(query = AG.sw.uniq, subject =
unique(WGRU[GenomicRanges::strand(WGRU) == "+"]))
GenomicRanges::mcols(AG.sw.uniq)[["uniq_intervals_minus"]] <-
GenomicRanges::countOverlaps(query = AG.sw.uniq, subject =
unique(WGRU[GenomicRanges::strand(WGRU) == "-"]))
GenomicRanges::mcols(AG.sw.primary.mult)[["primary_mult_piRNA_plus"]] <-
GenomicRanges::countOverlaps(query = AG.sw.primary.mult, subject =
WGRMP[GenomicRanges::strand(WGRMP) == "+"])
GenomicRanges::mcols(AG.sw.primary.mult)[["primary_mult_piRNA_minus"]] <-
GenomicRanges::countOverlaps(query = AG.sw.primary.mult, subject =
WGRMP[GenomicRanges::strand(WGRMP) == "-"])
GenomicRanges::mcols(AG.sw.secondary.mult)[["secondary_mult_piRNA_plus"]] <-
GenomicRanges::countOverlaps(query = AG.sw.secondary.mult, subject =
IN.ALIGNMENTS$multi.secondary[GenomicRanges::strand(IN.ALIGNMENTS$multi.secondary) ==
"+"])
GenomicRanges::mcols(AG.sw.secondary.mult)[["secondary_mult_piRNA_minus"]] <-
GenomicRanges::countOverlaps(query = AG.sw.secondary.mult, subject =
IN.ALIGNMENTS$multi.secondary[GenomicRanges::strand(IN.ALIGNMENTS$multi.secondary) == "-
"])
## Filter by minimum reads per window and reduce to new range
if (VERBOSITY>1) message(paste0("\t\tMIN.UNIQUE.ALIGNMENTS.PER.WINDOW >=
",MIN.UNIQUE.ALIGNMENTS.PER.WINDOW))
sw.UNIQ.PLUS <- AG.sw.uniq[(GenomicRanges::mcols(AG.sw.uniq)[["uniq_piRNA_plus"]] >=
MIN.UNIQUE.ALIGNMENTS.PER.WINDOW) & (GenomicRanges::mcols(AG.sw.uniq)
[["uniq_intervals_plus"]] >= MIN.UNIQUE.SEQUENCES.PER.WINDOW)]
sw.UNIQ.MINUS <- AG.sw.uniq[(GenomicRanges::mcols(AG.sw.uniq)[["uniq_piRNA_minus"]]
>= MIN.UNIQUE.ALIGNMENTS.PER.WINDOW) & (GenomicRanges::mcols(AG.sw.uniq)
[["uniq_intervals_minus"]] >= MIN.UNIQUE.SEQUENCES.PER.WINDOW)]
AG.sw.uniq<-NULL # clearing memory
GenomicRanges::strand(sw.UNIQ.PLUS) <- "+"; GenomicRanges::strand(sw.UNIQ.MINUS) <- "-"
sw.UNIQ.RED <- GenomicRanges::sort.GenomicRanges(GenomicRanges::reduce(c(sw.UNIQ.PLUS,sw.UNIQ.MINUS)))
sw.UNIQ.PLUS<-NULL #clearing memory
## Removing gaps between SEEDS
if (THRESHOLD.SEEDS.GAP>0){
GR.GAPS <- GenomicRanges::gaps(sw.UNIQ.RED)
GR.GAPS <- GenomicRanges::sort.GenomicRanges(GR.GAPS)
GR.GAPS.WIDER.THAN.MIN <- GR.GAPS[GenomicRanges::width(GR.GAPS) >=
THRESHOLD.SEEDS.GAP]
sw.UNIQ.RED <- GenomicRanges::gaps(GR.GAPS.WIDER.THAN.MIN) #gaps of gaps
sw.UNIQ.RED <- GenomicRanges::sort.GenomicRanges(sw.UNIQ.RED)
}
sw.UNIQ.RED<-
GenomicRanges::reduce(sw.UNIQ.RED[GenomicRanges::width(sw.UNIQ.RED)>=MIN.SEED.LENGTH])
if (VERBOSITY>1) message("Annotating seeds")
BM.SEEDS<-PICBannotate(sw.UNIQ.RED,IN.ALIGNMENTS, REFERENCE.GENOME =
REFERENCE.GENOME, LIBRARY.SIZE = LIBRARY.SIZE,
SEQ.LEVELS.STYLE = SEQ.LEVELS.STYLE, PROVIDE.NON.NORMALIZED =
TRUE)
if (VERBOSITY>1) message(paste("Removing seed with unique mapping coverage less
than", MIN.COVERED.SEED.LENGTH, 'nt'))
BM.SEEDS<-BM.SEEDS[GenomicRanges::mcols(BM.SEEDS)
[['width_covered_by_unique_alignments']]>=MIN.COVERED.SEED.LENGTH]
outputList<-list()
outputList[[uniqueonly]]<-BM.SEEDS
if (VERBOSITY>1) message(paste0("\t\tMIN.PRIMARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW
>= ",MIN.PRIMARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW))
sw.PRIMARY.MULT.PLUS <- AG.sw.primary.mult[GenomicRanges::mcols(AG.sw.primary.mult)
[["primary_mult_piRNA_plus"]] >= MIN.PRIMARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW]
sw.PRIMARY.MULT.MINUS <- AG.sw.primary.mult[GenomicRanges::mcols(AG.sw.primary.mult)
[["primary_mult_piRNA_minus"]] >= MIN.PRIMARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW]
GenomicRanges::strand(sw.PRIMARY.MULT.PLUS) <- "+";
GenomicRanges::strand(sw.PRIMARY.MULT.MINUS) <- "-"
sw.PRIMARY.MULT.PLUS.RED<-GenomicRanges::reduce(sw.PRIMARY.MULT.PLUS)
sw.PRIMARY.MULT.MINUS.RED<-GenomicRanges::reduce(sw.PRIMARY.MULT.MINUS)
sw.PRIMARY.MULT.PLUS.ANCHORED <- IRanges::subsetByOverlaps(x =
sw.PRIMARY.MULT.PLUS.RED, ranges = BM.SEEDS, minoverlap = MIN.OVERLAP)
sw.PRIMARY.MULT.MINUS.ANCHORED <- IRanges::subsetByOverlaps(x =
sw.PRIMARY.MULT.MINUS.RED, ranges = BM.SEEDS, minoverlap = MIN.OVERLAP)
sw.PRIMARY.MULT.RED.ANCHORED <- GenomicRanges::sort.GenomicRanges(GenomicRanges::reduce(c(sw.PRIMARY.MULT.PLUS.ANCHORED,sw.PRIMARY.MULT.MINUS.ANCHORED)))
sw.PRIMARY.MULT.RED.ANCHORED <- GenomicRanges::sort.GenomicRanges(GenomicRanges::reduce(c(sw.PRIMARY.MULT.RED.ANCHORED,BM.SEEDS)))
ONE.WINDOW.GR <- c(BM.SEEDS,sw.PRIMARY.MULT.RED.ANCHORED)
BM.CORES <- GenomicRanges::reduce(ONE.WINDOW.GR)
## Removing gaps between cores
if (THRESHOLD.CORES.GAP>0){
GR.GAPS <- GenomicRanges::gaps(BM.CORES)
GR.GAPS <- GenomicRanges::sort.GenomicRanges(GR.GAPS)
GR.GAPS.WIDER.THAN.MIN <- GR.GAPS[GenomicRanges::width(GR.GAPS) >= THRESHOLD.CORES.GAP]
BM.CORES <- GenomicRanges::gaps(GR.GAPS.WIDER.THAN.MIN) #gaps of gaps
BM.CORES <- GenomicRanges::sort.GenomicRanges(BM.CORES)
}
##Regions
if (VERBOSITY>1)
message(paste0("\t\tMIN.SECODNARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW >=
",MIN.SECONDARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW))
sw.SECONDARY.MULT.PLUS <- AG.sw.secondary.mult[GenomicRanges::mcols(AG.sw.secondary.mult)
[["secondary_mult_piRNA_plus"]] >= MIN.SECONDARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW]
sw.SECONDARY.MULT.MINUS <- AG.sw.secondary.mult[GenomicRanges::mcols(AG.sw.secondary.mult)
[["secondary_mult_piRNA_minus"]] >= MIN.SECONDARY.MULTIMAPPING.ALIGNMENTS.PER.WINDOW]
GenomicRanges::strand(sw.SECONDARY.MULT.PLUS) <- "+";
GenomicRanges::strand(sw.SECONDARY.MULT.MINUS) <- "-"
sw.SECONDARY.MULT.PLUS.RED<-GenomicRanges::reduce(sw.SECONDARY.MULT.PLUS)
sw.SECONDARY.MULT.MINUS.RED<-GenomicRanges::reduce(sw.SECONDARY.MULT.MINUS)
sw.SECONDARY.MULT.PLUS.ANCHORED <- IRanges::subsetByOverlaps(x = sw.SECONDARY.MULT.PLUS.RED, ranges = BM.CORES, minoverlap = MIN.OVERLAP)
sw.SECONDARY.MULT.MINUS.ANCHORED <- IRanges::subsetByOverlaps(x = sw.SECONDARY.MULT.MINUS.RED, ranges = BM.CORES, minoverlap = MIN.OVERLAP)
sw.SECONDARY.MULT.RED.ANCHORED <- GenomicRanges::sort.GenomicRanges(GenomicRanges::reduce(c(sw.SECONDARY.MULT.PLUS.ANCHORED,sw.SECONDARY.MULT.MINUS.ANCHORED)))
sw.SECONDARY.MULT.RED.ANCHORED <- GenomicRanges::sort.GenomicRanges(GenomicRanges::reduce(c(sw.SECONDARY.MULT.RED.ANCHORED,BM.CORES)))
CLUSTERS.GR <- c(BM.CORES,sw.SECONDARY.MULT.RED.ANCHORED)
CLUSTERS <- GenomicRanges::reduce(CLUSTERS.GR)
## Removing gaps between CCLUSTERS
if (THRESHOLD.CLUSTERS.GAP>0){
GR.GAPS <- GenomicRanges::gaps(CLUSTERS)
GR.GAPS <- GenomicRanges::sort.GenomicRanges(GR.GAPS)
GR.GAPS.WIDER.THAN.MIN <- GR.GAPS[GenomicRanges::width(GR.GAPS) >=
THRESHOLD.CLUSTERS.GAP]
CLUSTERS <- GenomicRanges::gaps(GR.GAPS.WIDER.THAN.MIN) #gaps of gaps
CLUSTERS <- GenomicRanges::sort.GenomicRanges(CLUSTERS)
}
if (VERBOSITY>0) message("\nBuilding ... STEP 2 ... Annotating & filtering\n")
## Annotate the cores
if (VERBOSITY>1) message("\tAnnotating cores and clusters")
outputList[[uniqueandprimary]]<-BM.CORES
outputList[[allalignments]]<-CLUSTERS
outputList<-PICBannotate(outputList, IN.ALIGNMENTS, , REFERENCE.GENOME =
REFERENCE.GENOME,
SEQ.LEVELS.STYLE = SEQ.LEVELS.STYLE, PROVIDE.NON.NORMALIZED
= TRUE, LIBRARY.SIZE = LIBRARY.SIZE)
## REPORT
if (VERBOSITY>1) message("\n\tClusters stats: ")
used_uniq <- sum(GenomicRanges::mcols(outputList[[allalignments]])[["uniq_reads"]])
used_uniq_piRNA <- round(((used_uniq/length(IN.ALIGNMENTS$unique))*100),3)
used_mult <- sum(GenomicRanges::mcols(outputList[[allalignments]])
[["multimapping_reads_primary_alignments"]])
used_mult_primary_piRNA <- round(((used_mult/length(WGRMP))*100),3)
if (VERBOSITY>1) {
message(paste0("\t\tAcomodated UNIQUE-mappers: ",used_uniq," (",used_uniq_piRNA,"
%)"))
message(paste0("\t\tAcomodated primary MULTI-mappers: ",used_mult,"
(",used_mult_primary_piRNA," %)"))
}
if (VERBOSITY>0) message("\nFINISHED.")
if (! PROVIDE.NON.NORMALIZED){ #removing stats hard to understand
for ( t in c(uniqueonly, uniqueandprimary, allalignments)){
GenomicRanges::mcols(outputList[[t]])[["uniq_reads"]] <- NULL
GenomicRanges::mcols(outputList[[t]])[["uniq_sequences"]] <- NULL
GenomicRanges::mcols(outputList[[t]])[["multimapping_reads_primary_alignments"]] <- NULL
GenomicRanges::mcols(outputList[[t]])[["all_reads_primary_alignments"]] <- NULL
GenomicRanges::mcols(outputList[[t]])[["width_covered_by_unique_alignments"]] <- NULL
}
}
return(outputList)
}

In [ ]:
#' @param EXCEL.FILE.NAME file name to import from
#'
#' @return list of annotated Granges objects named "seeds" for seeds,
#' "cores" for cores,
#' "clusters" for clusters
#' @export
#'
#' @examples PICBimportfromexcel(="~/piRNAclustersFromUFOsamples.Area51.xlsx")
EXCEL.FILE.NAME <- "/mnt/home3/miska/nm667/scratch/inProgress/mice_PiRNA/tmp/testpicb/piRNAclustersFromUFOsamples.Area51.xlsx"
PICBimportfromexcel <- function(EXCEL.FILE.NAME = NULL){
output=list()
output$seeds<-GenomicRanges::GRanges(openxlsx::read.xlsx(EXCEL.FILE.NAME, sheet = uniqueonly))
output$cores<-GenomicRanges::GRanges(openxlsx::read.xlsx(EXCEL.FILE.NAME, sheet = uniqueandprimary))
output$clusters<-GenomicRanges::GRanges(openxlsx::read.xlsx(EXCEL.FILE.NAME, sheet = uniqueandprimary))
return(output)
}